# 05 Final Load Prep

Assemble Tableau-ready tables from the two cleaned datasets.

**Outputs to `data/processed/`**
- `tableau_close_approaches_enriched.csv` — every close approach joined to NEA attributes (PHA flag, diameter, class, size category)
- `tableau_nea_catalogue.csv` — NEA catalogue trimmed to dashboard-relevant columns
- `tableau_yearly_summary.csv` — annual KPIs: total approaches, PHA approaches, closest pass, median distance, median velocity
- `tableau_top_threats.csv` — top 50 closest future approaches with hazard context


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

close = pd.read_csv(PROCESSED_DIR / 'close_approaches_clean.csv', parse_dates=['close_approach_date'])
nea   = pd.read_csv(PROCESSED_DIR / 'near_earth_asteroids_clean.csv', parse_dates=['first_obs', 'last_obs'])
print('close:', close.shape, '| nea:', nea.shape)


## 1. Enriched close approaches (join to NEA)

In [ ]:
nea_slim = nea[[
    'pdes', 'full_name', 'pha', 'h', 'diameter_km', 'diameter_m', 'size_category',
    'class', 'e', 'a', 'i', 'moid_au', 'moid_lunar_distances'
]].rename(columns={'full_name': 'nea_full_name'})

enriched = close.merge(nea_slim, left_on='designation', right_on='pdes', how='left')
enriched['pha'] = enriched['pha'].fillna(False).astype(bool)
enriched['hazard_flag'] = np.where(enriched['pha'], 'PHA', 'Non-PHA')

print('Matched to catalogue:', enriched['pdes'].notna().sum(), '/', len(enriched))
enriched.head(3)


In [ ]:
ENRICHED_OUT = PROCESSED_DIR / 'tableau_close_approaches_enriched.csv'
enriched.to_csv(ENRICHED_OUT, index=False)
print(f'Saved -> {ENRICHED_OUT} ({len(enriched):,} rows)')


## 2. NEA dashboard catalogue

In [ ]:
catalogue = nea[[
    'spkid', 'pdes', 'full_name', 'name', 'pha', 'h', 'diameter_km', 'diameter_m',
    'size_category', 'class', 'e', 'a', 'i', 'q', 'ad', 'per_y',
    'moid_au', 'moid_km', 'moid_lunar_distances',
    'first_obs', 'last_obs', 'observation_span_years', 'is_named'
]].copy()

CATALOGUE_OUT = PROCESSED_DIR / 'tableau_nea_catalogue.csv'
catalogue.to_csv(CATALOGUE_OUT, index=False)
print(f'Saved -> {CATALOGUE_OUT} ({len(catalogue):,} rows)')


## 3. Yearly KPI summary

In [ ]:
yearly = enriched.groupby('approach_year').agg(
    total_approaches=('designation', 'size'),
    unique_objects=('designation', 'nunique'),
    pha_approaches=('pha', 'sum'),
    closest_pass_ld=('dist_lunar', 'min'),
    median_distance_ld=('dist_lunar', 'median'),
    median_velocity_kms=('velocity_km_s', 'median'),
    max_velocity_kms=('velocity_km_s', 'max'),
).reset_index()

yearly['pha_share_pct'] = (yearly['pha_approaches'] / yearly['total_approaches'] * 100).round(2)

YEARLY_OUT = PROCESSED_DIR / 'tableau_yearly_summary.csv'
yearly.to_csv(YEARLY_OUT, index=False)
print(f'Saved -> {YEARLY_OUT}')
yearly.head()


## 4. Top-50 upcoming threats (future, closest)

In [ ]:
future = enriched[enriched['is_future'] == True].copy()
top_threats = (
    future.sort_values('dist_lunar')
    .head(50)[[
        'designation', 'nea_full_name', 'close_approach_date',
        'dist_lunar', 'dist_km', 'velocity_km_s',
        'diameter_km', 'size_category', 'hazard_flag', 'class'
    ]]
    .reset_index(drop=True)
)

THREATS_OUT = PROCESSED_DIR / 'tableau_top_threats.csv'
top_threats.to_csv(THREATS_OUT, index=False)
print(f'Saved -> {THREATS_OUT}')
top_threats.head(10)


## 5. Final manifest

In [ ]:
for p in sorted(PROCESSED_DIR.glob('*.csv')):
    print(f'{p.name:<50} {p.stat().st_size/1024:>9.1f} KB')


### Dashboard usage

- **Close approaches enriched** → map/timeline of every event, colored by `hazard_flag`
- **NEA catalogue** → filterable table of objects, sized by `diameter_km`
- **Yearly summary** → KPI cards + trend lines
- **Top threats** → highlighted list on the landing view